# Reading and Writing Tensors (import_data, export_data)

```
Copyright 2025 National Technology & Engineering Solutions of Sandia,
LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the
U.S. Government retains certain rights in this software.
```

The functions `export_data` and `import_data` write tensors, Kruskal tensors, and matrices to text files and read them back. The file format is compatible with the MATLAB Tensor Toolbox. All of the files in this tutorial are written to a temporary directory.

[API reference](../io.rst)

In [ ]:
from __future__ import annotations

import os
import tempfile

import numpy as np

import pyttb as ttb

## Overview
`export_data(data, filename)` supports a `tensor`, `sptensor`, `ktensor`, or a matrix (a two-dimensional `numpy.ndarray`). `import_data(filename)` reads the file and returns an object of the type recorded in the file.

### Setup
We create a temporary directory to hold the files.

In [ ]:
tmpdir = tempfile.TemporaryDirectory()
print(os.path.isdir(tmpdir.name))

## Round trips

### Dense tensor

In [ ]:
T = ttb.tensor(np.arange(24, dtype=float), shape=(3, 4, 2))
path = os.path.join(tmpdir.name, "tensor.tns")
ttb.export_data(T, path)
T_in = ttb.import_data(path)
print(type(T_in).__name__, T_in.shape)
T_in.isequal(T)

### Sparse tensor

In [ ]:
S = ttb.sptensor(
    np.array([[0, 0, 0], [1, 2, 1], [2, 3, 0]]),
    np.array([[1.5], [2.5], [3.5]]),
    shape=(3, 4, 2),
)
path = os.path.join(tmpdir.name, "sptensor.tns")
ttb.export_data(S, path)
S_in = ttb.import_data(path)
print(type(S_in).__name__, S_in.shape)
S_in.isequal(S)

### Kruskal tensor
The weights are stored along with the factor matrices.

In [ ]:
rng = np.random.default_rng(0)
K = ttb.ktensor(
    [rng.random((3, 2)), rng.random((4, 2)), rng.random((2, 2))],
    weights=np.array([2.0, 1.0]),
)
path = os.path.join(tmpdir.name, "ktensor.tns")
ttb.export_data(K, path)
K_in = ttb.import_data(path)
print(type(K_in).__name__, K_in.shape, K_in.weights)
K_in.isequal(K)

### Matrix
A two-dimensional array is stored as a matrix and is read back as an array.

In [ ]:
A = rng.random((3, 4))
path = os.path.join(tmpdir.name, "matrix.tns")
ttb.export_data(A, path)
A_in = ttb.import_data(path)
print(type(A_in).__name__, A_in.shape)
np.allclose(A_in, A)

## File format
Each file starts with a line giving the type of data (`tensor`, `sptensor`, `ktensor`, or `matrix`), a line with the number of dimensions, and a line with the shape. The remaining lines depend on the type:
* `tensor`: the values, one per line, in column-major order.
* `matrix`: the values, one per line, in row-major order.
* `sptensor`: the number of nonzeros, then one line per nonzero with its subscripts followed by its value.
* `ktensor`: the number of components, a line with the weights, and then each factor matrix written as a `matrix` (type, dimensions, shape, values).

As an example, here is the file for a small sparse tensor.

In [ ]:
S_small = ttb.sptensor(
    np.array([[0, 0, 0], [1, 2, 1]]), np.array([[1.0], [2.0]]), shape=(2, 3, 2)
)
path = os.path.join(tmpdir.name, "small.tns")
ttb.export_data(S_small, path, fmt_data="%.1f")
print(open(path).read())

## Options
### `fmt_data` and `fmt_weights`
The numerical format of the values defaults to `"%.16e"`, which preserves full double precision. A shorter format such as `"%.4f"` gives smaller files at the cost of precision. For a `ktensor`, `fmt_weights` sets the format of the weights separately.

In [ ]:
path_full = os.path.join(tmpdir.name, "full_precision.tns")
path_short = os.path.join(tmpdir.name, "short.tns")
ttb.export_data(T, path_full)
ttb.export_data(T, path_short, fmt_data="%.2f")
print(os.path.getsize(path_full), os.path.getsize(path_short))

rounded = ttb.import_data(path_short)
print(np.allclose(rounded.data, T.data, atol=0.01))

### `index_base`
Sparse tensor subscripts are stored in the file with a configurable base. The default `index_base=1` matches MATLAB, while `index_base=0` matches Python's zero-based indexing. The same value must be used for reading and writing. Here we write the same tensor with each base and look at the line of the first nonzero in the file.

In [ ]:
path_one = os.path.join(tmpdir.name, "base1.tns")
path_zero = os.path.join(tmpdir.name, "base0.tns")
ttb.export_data(S_small, path_one)  # index_base=1 (MATLAB style)
ttb.export_data(S_small, path_zero, index_base=0)
print(open(path_one).read().splitlines()[4])
print(open(path_zero).read().splitlines()[4])
print(ttb.import_data(path_one).isequal(S_small))
print(ttb.import_data(path_zero, index_base=0).isequal(S_small))

### Cleaning up
Remove the temporary directory and its files.

In [ ]:
tmpdir.cleanup()
os.path.exists(tmpdir.name)

## Summary
Use `export_data` and `import_data` to save tensors to text files and read them back, with `fmt_data`, `fmt_weights`, and `index_base` to control the format. See the [input/output](../io.rst) API page for details.